# Last.fm bench

Compare embedding recommendations against Last.fm batch results. Point `BENCH_PATH` at any `lastfm_recommendations_*.csv` file (long format with `rank` column).

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../datasets")
BENCH_PATH = DATA_DIR / "lastfm_recommendations_1k_random.csv"
RECS_PATH = DATA_DIR / "recommendations_album_level_avg_embeddings.parquet"

## Recommendations


In [ ]:
recs = pd.read_parquet(RECS_PATH)

In [ ]:
recs = recs.drop(columns=["rec_length_flag"])
recs.head(5)

## Baseline

In [ ]:
df = pd.read_csv(BENCH_PATH)
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")

# keep successful recommendations only
df = df[df["status"] == "ok"].reset_index(drop=True)

n_albums = df["album_id"].nunique()
print(f"File: {BENCH_PATH.name}")
print(f"Strategy: {df['strategy'].iloc[0]}  |  Albums: {n_albums:,}  |  Rec rows: {len(df):,}")
df.head(5)

In [ ]:
baseline = df.rename(columns={
    "artist": "query_artist",
    "album":  "query_album",
})[["query_artist", "query_album", "rec_artist", "rec_album", "score", "rank"]].copy()

print(f"Baseline rows: {len(baseline):,}")
baseline.head(5)

# Comparisons

## Repeated Albums

In [ ]:
def repeated_albums(df, label, rec_col="rec_album"):
    counts = df[rec_col].str.strip().str.lower().value_counts()
    n_total = len(df)
    n_unique = counts.size
    n_repeated = (counts > 1).sum()
    print(f"[{label}]")
    print(f"  Total:    {n_total:,}")
    print(f"  Unique:   {n_unique:,}  ({n_unique / n_total:.1%})")
    print(f"  Repeated: {n_repeated:,}  ({n_repeated / n_unique:.1%} of unique)")
    print()


repeated_albums(recs[recs["rank"] == 1], "embedding recs")
repeated_albums(baseline[baseline["rank"] == 1], "last.fm baseline")

## Score distribution

In [ ]:
import matplotlib.pyplot as plt

rank1_recs = recs[recs["rank"] == 1]
rank1_base = baseline[baseline["rank"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, s, label in zip(
    axes,
    [rank1_recs["score"], rank1_base["score"]],
    ["Embedding recs", "Last.fm baseline"],
):
    ax.hist(s.dropna(), bins=40, color="#4472C4", edgecolor="white")
    ax.axvline(s.mean(), color="crimson", ls="--", label=f"mean={s.mean():.3f}")
    ax.set(title=label, xlabel="Score", ylabel="Count")
    ax.legend()
plt.tight_layout()
plt.show()

## Reciprocity

In [ ]:
def reciprocity(df, label):
    """If A recommends B, does B recommend A back?"""
    norm = lambda s: s.str.strip().str.lower()
    edges = set(zip(norm(df["query_artist"]) + "::" + norm(df["query_album"]),
                    norm(df["rec_artist"])   + "::" + norm(df["rec_album"])))
    reciprocal = sum(1 for a, b in edges if (b, a) in edges)
    rate = reciprocal / len(edges) if edges else 0
    print(f"[{label}]")
    print(f"  Edges:       {len(edges):,}")
    print(f"  Reciprocal:  {reciprocal:,}")
    print(f"  Rate:        {rate:.1%}")
    print()


reciprocity(recs[recs["rank"] == 1], "embedding recs")
reciprocity(baseline[baseline["rank"] == 1], "last.fm baseline")